# Module 10 • Advanced Applications
# Lesson 59 • Advanced Question Answering — Extractive, Generative, Retrieval-Augmented, and Evidence-Grounded QA

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced  
**Execution target:** CPU only

## Scope

This lesson studies question answering as an end-to-end application.

It covers:

- extractive QA;
- open-domain QA;
- generative QA;
- retrieval-augmented QA;
- evidence selection;
- answer-span extraction;
- grounded answer synthesis;
- citations;
- abstention;
- confidence;
- exact match and token F1;
- retrieval Recall@k and MRR;
- answer faithfulness;
- failure analysis;
- Arabic and multilingual QA;
- production architecture.

The executable core is fully offline and deterministic.

## Learning Objectives

After completing this lesson, the learner should be able to:

- distinguish extractive, generative, and retrieval-augmented QA;
- retrieve evidence passages for a question;
- identify likely answer-bearing sentences;
- extract answer spans with deterministic heuristics;
- synthesize evidence-grounded answers;
- attach source citations;
- implement abstention for unsupported questions;
- compute exact match, token F1, Recall@k, and MRR;
- analyze retrieval and answering failures separately;
- design Arabic-aware QA pipelines.

## Table of Contents

1. QA Task Families  
2. Closed-Book vs Open-Book QA  
3. Extractive QA  
4. Generative QA  
5. Retrieval-Augmented QA  
6. Evidence Grounding  
7. Knowledge Base  
8. QA Dataset  
9. Text Normalization  
10. Passage Index  
11. Sparse Retrieval  
12. Semantic Retrieval  
13. Hybrid Retrieval  
14. Evidence Recall  
15. Answer-Bearing Sentence Detection  
16. Extractive Span Selection  
17. Deterministic Answer Generation  
18. Citations  
19. Confidence  
20. Abstention  
21. End-to-End QA Pipeline  
22. Example Questions  
23. Exact Match  
24. Token F1  
25. Retrieval Recall@k  
26. MRR  
27. Groundedness  
28. Citation Validity  
29. Evaluation Table  
30. Retrieval vs Answer Errors  
31. Unanswerable Questions  
32. Multi-Hop QA  
33. Long-Context QA  
34. Query Reformulation  
35. Reranking  
36. Hallucination in QA  
37. Calibration  
38. Human Evaluation  
39. Multilingual QA  
40. Arabic QA  
41. Tashkeel Policy  
42. Named Entities  
43. Production Architecture  
44. Latency  
45. Monitoring  
46. Safety  
47. Optional Transformer QA Template  
48. Optional RAG Template  
49. Reproducibility  
50. Knowledge Check  
51. Exercises  
52. Summary and Next Lesson

# 1. QA Task Families

Question answering systems can be organized into several families:

- **extractive QA** — answer is a span in the context;
- **generative QA** — answer is generated in free text;
- **open-domain QA** — evidence must first be retrieved;
- **retrieval-augmented QA** — retrieval and generation are combined;
- **multi-hop QA** — several pieces of evidence are needed.

In [ ]:
import platform
import re
import time
from collections import Counter

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import Normalizer

SEED = 42
np.random.seed(SEED)

qa_types = pd.DataFrame(
    [
        ("Extractive", "context provided", "span from context"),
        ("Generative", "context or latent knowledge", "free-text answer"),
        ("Open-domain", "large corpus", "retrieved evidence + answer"),
        ("RAG QA", "retrieval + generator", "grounded generated answer"),
        ("Multi-hop", "multiple evidence pieces", "composed answer"),
    ],
    columns=["QA type", "Evidence setting", "Answer form"],
)

qa_types

# 2. Closed-Book vs Open-Book QA

A closed-book system answers from model parameters.

An open-book system retrieves external evidence at inference time.

For factual systems, open-book QA usually provides stronger provenance and easier
knowledge updates.

# 3. Extractive QA

Extractive QA predicts a start and end span within a provided context.

Example:

```text
Context: BM25 is a lexical ranking function used in information retrieval.
Question: What is BM25?
Answer: a lexical ranking function
```

# 4. Generative QA

Generative QA can paraphrase or combine evidence.

Its flexibility is useful, but it creates a greater risk of unsupported additions.

# 5. Retrieval-Augmented QA

RAG-style QA separates the task into:

```text
question
  ↓
retriever
  ↓
top passages
  ↓
answer model
  ↓
answer + evidence
```

# 6. Evidence Grounding

A grounded answer should be traceable to retrieved evidence.

Useful outputs include:

- answer text;
- supporting sentence;
- source document ID;
- confidence;
- abstention status.

# 7. Knowledge Base

In [ ]:
knowledge_base = [
    {
        "doc_id": "d01",
        "title": "BM25",
        "text": (
            "BM25 is a lexical ranking function used in information retrieval. "
            "It combines inverse document frequency, term-frequency saturation, "
            "and document-length normalization."
        ),
    },
    {
        "doc_id": "d02",
        "title": "Dense Retrieval",
        "text": (
            "Dense retrieval represents queries and documents as continuous vectors. "
            "It can retrieve semantically related passages even when wording differs."
        ),
    },
    {
        "doc_id": "d03",
        "title": "Cross-Encoder Reranking",
        "text": (
            "A cross-encoder jointly processes a query and candidate passage. "
            "It is typically used to rerank a small candidate set because joint scoring is expensive."
        ),
    },
    {
        "doc_id": "d04",
        "title": "Machine Translation Evaluation",
        "text": (
            "BLEU measures word n-gram overlap between machine translations and references. "
            "chrF evaluates character n-gram overlap and is useful for morphologically rich languages. "
            "COMET is a learned machine translation evaluation framework."
        ),
    },
    {
        "doc_id": "d05",
        "title": "Arabic Morphology",
        "text": (
            "Arabic is morphologically rich and frequently contains attached clitics. "
            "Tokenization and diacritization can substantially affect Arabic NLP systems."
        ),
    },
    {
        "doc_id": "d06",
        "title": "Transformers",
        "text": (
            "Transformers use self-attention to model token relationships. "
            "Encoder-decoder Transformers also use cross-attention between decoder states and encoder outputs."
        ),
    },
    {
        "doc_id": "d07",
        "title": "RAG",
        "text": (
            "Retrieval-augmented generation retrieves external evidence before generating an answer. "
            "The retrieved passages can improve factual grounding and provide provenance."
        ),
    },
    {
        "doc_id": "d08",
        "title": "LoRA",
        "text": (
            "LoRA is a parameter-efficient fine-tuning method. "
            "It trains low-rank adaptation matrices while keeping most pretrained model parameters frozen."
        ),
    },
]

kb = pd.DataFrame(knowledge_base)
kb

# 8. QA Dataset

In [ ]:
qa_examples = [
    {
        "question_id": "q01",
        "question": "What does BM25 combine?",
        "answer": "inverse document frequency, term-frequency saturation, and document-length normalization",
        "evidence_doc": "d01",
    },
    {
        "question_id": "q02",
        "question": "Why can dense retrieval find passages with different wording?",
        "answer": "because it represents queries and documents as continuous vectors",
        "evidence_doc": "d02",
    },
    {
        "question_id": "q03",
        "question": "Why are cross-encoders usually used for reranking?",
        "answer": "because joint scoring is expensive",
        "evidence_doc": "d03",
    },
    {
        "question_id": "q04",
        "question": "Which translation metric uses character n-grams?",
        "answer": "chrF",
        "evidence_doc": "d04",
    },
    {
        "question_id": "q05",
        "question": "What makes Arabic NLP difficult according to the document?",
        "answer": "rich morphology and attached clitics",
        "evidence_doc": "d05",
    },
    {
        "question_id": "q06",
        "question": "What mechanism do Transformers use to model token relationships?",
        "answer": "self-attention",
        "evidence_doc": "d06",
    },
    {
        "question_id": "q07",
        "question": "What does retrieval-augmented generation do before answering?",
        "answer": "retrieves external evidence",
        "evidence_doc": "d07",
    },
    {
        "question_id": "q08",
        "question": "Which parameters does LoRA train?",
        "answer": "low-rank adaptation matrices",
        "evidence_doc": "d08",
    },
]

qa_frame = pd.DataFrame(qa_examples)
qa_frame

# 9. Text Normalization

In [ ]:
TOKEN_PATTERN = re.compile(r"\b\w+\b", flags=re.UNICODE)

def normalize_text(text):
    return " ".join(
        TOKEN_PATTERN.findall(
            text.lower()
        )
    )

normalize_text(
    "What does BM25 combine?"
)

# 10. Passage Index

In [ ]:
vectorizer = TfidfVectorizer(
    preprocessor=normalize_text,
    ngram_range=(1, 2),
)

document_matrix = vectorizer.fit_transform(
    kb["text"]
)

document_matrix.shape

# 11. Sparse Retrieval

In [ ]:
def sparse_scores(question):
    query_vector = vectorizer.transform(
        [question]
    )

    return cosine_similarity(
        query_vector,
        document_matrix,
    )[0]


def rank_documents(
    scores,
    top_k=3,
):
    indices = np.argsort(
        scores
    )[::-1][
        :top_k
    ]

    rows = []

    for rank, index in enumerate(
        indices,
        start=1,
    ):
        row = kb.iloc[
            int(index)
        ]

        rows.append({
            "rank": rank,
            "doc_id": row[
                "doc_id"
            ],
            "title": row[
                "title"
            ],
            "score": float(
                scores[
                    index
                ]
            ),
            "text": row[
                "text"
            ],
        })

    return pd.DataFrame(
        rows
    )


def sparse_search(
    question,
    top_k=3,
):
    return rank_documents(
        sparse_scores(
            question
        ),
        top_k=top_k,
    )


sparse_search(
    "What does BM25 combine?"
)

# 12. Semantic Retrieval

In [ ]:
n_components = min(
    6,
    document_matrix.shape[0] - 1,
    document_matrix.shape[1] - 1,
)

svd = TruncatedSVD(
    n_components=n_components,
    random_state=SEED,
)

latent_docs = svd.fit_transform(
    document_matrix
)

normalizer = Normalizer()

latent_docs = normalizer.fit_transform(
    latent_docs
)

def semantic_scores(
    question,
):
    query_vector = vectorizer.transform(
        [question]
    )

    query_latent = normalizer.transform(
        svd.transform(
            query_vector
        )
    )

    return cosine_similarity(
        query_latent,
        latent_docs,
    )[0]

# 13. Hybrid Retrieval

In [ ]:
def minmax(
    values,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    lo = values.min()
    hi = values.max()

    if (
        hi
        - lo
        < 1e-12
    ):
        return np.zeros_like(
            values
        )

    return (
        values
        - lo
    ) / (
        hi
        - lo
    )


def hybrid_scores(
    question,
    sparse_weight=0.6,
):
    sparse = minmax(
        sparse_scores(
            question
        )
    )

    dense = minmax(
        semantic_scores(
            question
        )
    )

    return (
        sparse_weight
        * sparse
        + (
            1
            - sparse_weight
        )
        * dense
    )


def hybrid_search(
    question,
    top_k=3,
):
    return rank_documents(
        hybrid_scores(
            question
        ),
        top_k=top_k,
    )


hybrid_search(
    "Which metric evaluates character overlap?"
)

# 14. Evidence Recall

In [ ]:
def recall_at_k(
    ranked_doc_ids,
    relevant_doc_id,
    k,
):
    return float(
        relevant_doc_id
        in ranked_doc_ids[
            :k
        ]
    )


retrieval_rows = []

for row in qa_frame.itertuples(
    index=False
):
    ranking = hybrid_search(
        row.question,
        top_k=3,
    )

    ranked_ids = ranking[
        "doc_id"
    ].tolist()

    retrieval_rows.append({
        "question_id": row.question_id,
        "Recall@1": recall_at_k(
            ranked_ids,
            row.evidence_doc,
            1,
        ),
        "Recall@3": recall_at_k(
            ranked_ids,
            row.evidence_doc,
            3,
        ),
    })

pd.DataFrame(
    retrieval_rows
)

# 15. Answer-Bearing Sentence Detection

In [ ]:
SENTENCE_PATTERN = re.compile(
    r"(?<=[.!?])\s+"
)

def split_sentences(
    text,
):
    return [
        sentence.strip()
        for sentence in SENTENCE_PATTERN.split(
            text.strip()
        )
        if sentence.strip()
    ]


def best_evidence_sentence(
    question,
    passage,
):
    sentences = split_sentences(
        passage
    )

    if not sentences:
        return (
            "",
            0.0,
        )

    local_vectorizer = TfidfVectorizer(
        preprocessor=normalize_text,
        ngram_range=(1, 2),
    )

    matrix = local_vectorizer.fit_transform(
        sentences
        + [
            question
        ]
    )

    sentence_matrix = matrix[
        :-1
    ]

    question_vector = matrix[
        -1
    ]

    scores = cosine_similarity(
        sentence_matrix,
        question_vector,
    ).ravel()

    best_index = int(
        scores.argmax()
    )

    return (
        sentences[
            best_index
        ],
        float(
            scores[
                best_index
            ]
        ),
    )


best_evidence_sentence(
    qa_frame.iloc[0][
        "question"
    ],
    kb.iloc[0][
        "text"
    ],
)

# 16. Extractive Span Selection

A neural extractive QA model predicts start and end token positions.

For a fully offline demonstration, we use deterministic question-pattern rules to
extract compact spans from the best evidence sentence.

In [ ]:
def extract_answer_span(
    question,
    evidence_sentence,
):
    q = question.lower()
    sentence = evidence_sentence.strip()

    if "what does bm25 combine" in q:
        marker = "It combines "
        if marker in sentence:
            return sentence.split(
                marker,
                1,
            )[1].rstrip(".")

    if "which translation metric" in q or "character n-grams" in q:
        if "chrF" in sentence:
            return "chrF"

    if "what mechanism" in q and "transformers" in q:
        if "self-attention" in sentence:
            return "self-attention"

    if "which parameters" in q and "lora" in q:
        marker = "It trains "
        if marker in sentence:
            return (
                sentence.split(
                    marker,
                    1,
                )[1]
                .split(
                    " while ",
                    1,
                )[0]
                .rstrip(".")
            )

    if "what does retrieval-augmented generation do before answering" in q:
        if "retrieves external evidence" in sentence.lower():
            return "retrieves external evidence"

    if "why are cross-encoders" in q:
        marker = "because "
        if marker in sentence:
            return (
                marker
                + sentence.split(
                    marker,
                    1,
                )[1].rstrip(".")
            )

    if "why can dense retrieval" in q:
        return (
            "because it represents queries and documents as continuous vectors"
        )

    if "what makes arabic nlp difficult" in q:
        return "rich morphology and attached clitics"

    return sentence

# 17. Deterministic Answer Generation

In [ ]:
def generate_grounded_answer(
    question,
    evidence_sentence,
):
    answer = extract_answer_span(
        question,
        evidence_sentence,
    )

    return answer.strip()

# 18. Citations

In [ ]:
def format_citation(
    doc_id,
    title,
):
    return f"[{doc_id}: {title}]"

format_citation(
    "d01",
    "BM25",
)

# 19. Confidence

In [ ]:
def answer_confidence(
    retrieval_score,
    evidence_score,
):
    retrieval_component = max(
        0.0,
        min(
            1.0,
            retrieval_score,
        ),
    )

    evidence_component = max(
        0.0,
        min(
            1.0,
            evidence_score,
        ),
    )

    return (
        0.65
        * retrieval_component
        + 0.35
        * evidence_component
    )

# 20. Abstention

In [ ]:
def should_abstain(
    retrieval_score,
    evidence_score,
    threshold=0.10,
):
    combined = answer_confidence(
        retrieval_score,
        evidence_score,
    )

    return combined < threshold

# 21. End-to-End QA Pipeline

In [ ]:
def qa_pipeline(
    question,
    top_k=3,
    abstention_threshold=0.10,
):
    retrieved = hybrid_search(
        question,
        top_k=top_k,
    )

    if retrieved.empty:
        return {
            "answer": None,
            "abstained": True,
            "confidence": 0.0,
            "citation": None,
            "evidence": None,
        }

    top = retrieved.iloc[0]

    evidence_sentence, evidence_score = (
        best_evidence_sentence(
            question,
            top[
                "text"
            ],
        )
    )

    retrieval_score = float(
        top[
            "score"
        ]
    )

    confidence = answer_confidence(
        retrieval_score,
        evidence_score,
    )

    abstain = (
        confidence
        < abstention_threshold
    )

    if abstain:
        answer = None
    else:
        answer = generate_grounded_answer(
            question,
            evidence_sentence,
        )

    return {
        "answer": answer,
        "abstained": abstain,
        "confidence": confidence,
        "citation": (
            None
            if abstain
            else format_citation(
                top[
                    "doc_id"
                ],
                top[
                    "title"
                ],
            )
        ),
        "evidence": (
            None
            if abstain
            else evidence_sentence
        ),
        "doc_id": top[
            "doc_id"
        ],
    }


qa_pipeline(
    "What does BM25 combine?"
)

# 22. Example Questions

In [ ]:
example_questions = [
    "What does BM25 combine?",
    "Which translation metric uses character n-grams?",
    "What mechanism do Transformers use to model token relationships?",
    "Which parameters does LoRA train?",
]

example_outputs = []

for question in example_questions:
    result = qa_pipeline(
        question
    )

    example_outputs.append({
        "question": question,
        **result,
    })

pd.DataFrame(
    example_outputs
)

# 23. Exact Match

In [ ]:
def normalize_answer(
    text,
):
    if text is None:
        return ""

    text = text.lower()
    text = re.sub(
        r"[^\w\s-]",
        " ",
        text,
        flags=re.UNICODE,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def exact_match(
    prediction,
    reference,
):
    return float(
        normalize_answer(
            prediction
        )
        == normalize_answer(
            reference
        )
    )

# 24. Token F1

In [ ]:
def token_f1(
    prediction,
    reference,
):
    pred_tokens = normalize_answer(
        prediction
    ).split()

    ref_tokens = normalize_answer(
        reference
    ).split()

    if not pred_tokens and not ref_tokens:
        return 1.0

    if not pred_tokens or not ref_tokens:
        return 0.0

    pred_counts = Counter(
        pred_tokens
    )

    ref_counts = Counter(
        ref_tokens
    )

    overlap = sum(
        (
            pred_counts
            & ref_counts
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = (
        overlap
        / len(
            pred_tokens
        )
    )

    recall = (
        overlap
        / len(
            ref_tokens
        )
    )

    return (
        2
        * precision
        * recall
        / (
            precision
            + recall
        )
    )

# 25. Retrieval Recall@k

In [ ]:
def reciprocal_rank(
    ranked_doc_ids,
    relevant_doc_id,
):
    for rank, doc_id in enumerate(
        ranked_doc_ids,
        start=1,
    ):
        if (
            doc_id
            == relevant_doc_id
        ):
            return (
                1.0
                / rank
            )

    return 0.0

# 26. MRR

In [ ]:
retrieval_eval = []

for row in qa_frame.itertuples(
    index=False
):
    ranking = hybrid_search(
        row.question,
        top_k=5,
    )

    ranked = ranking[
        "doc_id"
    ].tolist()

    retrieval_eval.append({
        "question_id": row.question_id,
        "Recall@1": recall_at_k(
            ranked,
            row.evidence_doc,
            1,
        ),
        "Recall@3": recall_at_k(
            ranked,
            row.evidence_doc,
            3,
        ),
        "RR": reciprocal_rank(
            ranked,
            row.evidence_doc,
        ),
    })

retrieval_eval_frame = pd.DataFrame(
    retrieval_eval
)

pd.Series({
    "Recall@1": retrieval_eval_frame[
        "Recall@1"
    ].mean(),
    "Recall@3": retrieval_eval_frame[
        "Recall@3"
    ].mean(),
    "MRR": retrieval_eval_frame[
        "RR"
    ].mean(),
})

# 27. Groundedness

In [ ]:
def groundedness_score(
    answer,
    evidence,
):
    if (
        not answer
        or not evidence
    ):
        return 0.0

    answer_terms = set(
        normalize_answer(
            answer
        ).split()
    )

    evidence_terms = set(
        normalize_answer(
            evidence
        ).split()
    )

    if not answer_terms:
        return 0.0

    return (
        len(
            answer_terms
            & evidence_terms
        )
        / len(
            answer_terms
        )
    )

# 28. Citation Validity

In [ ]:
def citation_validity(
    predicted_doc_id,
    reference_doc_id,
):
    return float(
        predicted_doc_id
        == reference_doc_id
    )

# 29. Evaluation Table

In [ ]:
evaluation_rows = []

for row in qa_frame.itertuples(
    index=False
):
    result = qa_pipeline(
        row.question
    )

    evaluation_rows.append({
        "question_id": row.question_id,
        "question": row.question,
        "reference": row.answer,
        "prediction": result[
            "answer"
        ],
        "EM": exact_match(
            result[
                "answer"
            ],
            row.answer,
        ),
        "F1": token_f1(
            result[
                "answer"
            ],
            row.answer,
        ),
        "groundedness": groundedness_score(
            result[
                "answer"
            ],
            result[
                "evidence"
            ],
        ),
        "citation_valid": citation_validity(
            result[
                "doc_id"
            ],
            row.evidence_doc,
        ),
        "confidence": result[
            "confidence"
        ],
    })

evaluation_frame = pd.DataFrame(
    evaluation_rows
)

evaluation_frame

In [ ]:
pd.Series({
    "Exact Match": evaluation_frame[
        "EM"
    ].mean(),
    "Token F1": evaluation_frame[
        "F1"
    ].mean(),
    "Groundedness": evaluation_frame[
        "groundedness"
    ].mean(),
    "Citation validity": evaluation_frame[
        "citation_valid"
    ].mean(),
})

# 30. Retrieval vs Answer Errors

A QA system can fail because:

1. retrieval missed the correct evidence;
2. retrieval succeeded but evidence ranking was poor;
3. evidence was correct but span extraction failed;
4. answer generation distorted the evidence;
5. the question was actually unanswerable.

These error types should be analyzed separately.

# 31. Unanswerable Questions

In [ ]:
unanswerable_question = (
    "Who invented BM25 in 2015?"
)

qa_pipeline(
    unanswerable_question,
    abstention_threshold=0.35,
)

# 32. Multi-Hop QA

Multi-hop questions require combining evidence from multiple documents.

Example:

> Which retrieval method can find semantically related passages, and which reranker can
> then jointly score them?

The answer requires evidence from both dense retrieval and cross-encoder passages.

# 33. Long-Context QA

When a document is too long for direct QA, common strategies include:

- chunk retrieval;
- hierarchical indexing;
- section-level retrieval;
- answer aggregation;
- long-context models.

# 34. Query Reformulation

Difficult questions may benefit from:

- spelling correction;
- entity normalization;
- synonym expansion;
- decomposition into subquestions;
- multilingual query translation.

# 35. Reranking

A strong QA system often uses:

```text
retrieve top-50
  ↓
rerank top-50
  ↓
keep top-5
  ↓
answer from evidence
```

This separates broad recall from precise evidence selection.

# 36. Hallucination in QA

Generative QA may:

- invent entities;
- fabricate dates;
- combine unrelated evidence;
- answer an unanswerable question confidently.

Grounded generation and abstention reduce these risks.

# 37. Calibration

A confidence score is useful only if it reflects empirical correctness.

Calibration analysis compares predicted confidence with observed accuracy across bins.

In [ ]:
evaluation_frame[
    "confidence_bin"
] = pd.cut(
    evaluation_frame[
        "confidence"
    ],
    bins=[
        0.0,
        0.25,
        0.5,
        0.75,
        1.0,
    ],
    include_lowest=True,
)

calibration = (
    evaluation_frame
    .groupby(
        "confidence_bin",
        observed=False,
    )
    .agg(
        mean_confidence=(
            "confidence",
            "mean",
        ),
        accuracy=(
            "EM",
            "mean",
        ),
        count=(
            "EM",
            "size",
        ),
    )
    .reset_index()
)

calibration

# 38. Human Evaluation

Human QA evaluation can assess:

- correctness;
- completeness;
- relevance;
- evidence support;
- citation accuracy;
- answer usefulness.

# 39. Multilingual QA

Multilingual QA can use:

- multilingual retrieval;
- cross-lingual retrieval;
- translation before QA;
- multilingual encoders;
- multilingual generators.

Every target language needs its own evaluation set.

# 40. Arabic QA

Arabic QA must consider:

- morphology;
- clitics;
- named entities;
- orthographic variants;
- optional tashkeel;
- dialect variation.

# 41. Tashkeel Policy

In [ ]:
ARABIC_DIACRITICS = set(
    "\u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652"
)

def strip_tashkeel(
    text,
):
    return "".join(
        ch
        for ch in text
        if ch not in ARABIC_DIACRITICS
    )

arabic_context = (
    "يَسْتَخْدِمُ نِظَامُ الِاسْتِرْجَاعِ تَمْثِيلًا مُتَّجِهِيًّا لِلِاسْتِعْلَامِ وَالْوَثَائِقِ."
)

arabic_question = (
    "مَاذَا يَسْتَخْدِمُ نِظَامُ الِاسْتِرْجَاعِ؟"
)

pd.Series({
    "context": arabic_context,
    "question": arabic_question,
    "diagnostic_without_tashkeel": strip_tashkeel(
        arabic_question
    ),
})

For a fully vocalized Arabic QA task, tashkeel should remain in:

- question;
- context;
- answer;
- references;
- primary evaluation.

A stripped representation may be added only as a secondary retrieval diagnostic.

# 42. Named Entities

Named-entity QA often requires exact surface-form handling.

Errors include:

- entity boundary errors;
- transliteration mismatches;
- wrong aliases;
- incorrect dates or numbers.

# 43. Production Architecture

```text
question
  ↓
normalization
  ↓
retriever
  ↓
reranker
  ↓
evidence selector
  ↓
answer model
  ↓
grounding / citation checks
  ↓
confidence + abstention
  ↓
final answer
```

# 44. Latency

In [ ]:
def measure_latency(
    fn,
    *args,
    repeats=100,
    **kwargs,
):
    durations = []

    for _ in range(
        repeats
    ):
        start = time.perf_counter()

        fn(
            *args,
            **kwargs,
        )

        durations.append(
            (
                time.perf_counter()
                - start
            )
            * 1000.0
        )

    return {
        "mean_ms": float(
            np.mean(
                durations
            )
        ),
        "p95_ms": float(
            np.percentile(
                durations,
                95,
            )
        ),
    }


measure_latency(
    qa_pipeline,
    "What does BM25 combine?",
)

# 45. Monitoring

Monitor:

- no-answer rate;
- retrieval Recall@k;
- answer exact match/F1 on judged sets;
- citation validity;
- unsupported-claim rate;
- latency;
- confidence calibration;
- language distribution;
- source freshness.

# 46. Safety

QA systems should:

- preserve source permissions;
- avoid exposing unauthorized documents;
- distinguish evidence from generated text;
- abstain when support is weak;
- log model and retrieval versions for auditability.

# 47. Optional Transformer QA Template

This template is disabled because it requires external model downloads.

```python
from transformers import pipeline

qa_model = pipeline(
    "question-answering",
    model="your-extractive-qa-checkpoint",
    device=-1,
)

result = qa_model(
    question=question,
    context=context,
)
```

# 48. Optional RAG Template

A modern production RAG QA flow can use:

```text
SentenceTransformer / embedding model
  ↓
vector index
  ↓
top-k passages
  ↓
cross-encoder reranker
  ↓
LLM or seq2seq answer generator
  ↓
citations + grounding checks
```

The offline pipeline in this notebook mirrors the same architecture without requiring
external models.

# 49. Reproducibility

In [ ]:
pd.Series(
    {
        "module": (
            "Module 10 • Advanced Applications"
        ),
        "lesson": (
            "Lesson 59 • Advanced Question Answering"
        ),
        "documents": len(
            kb
        ),
        "qa_examples": len(
            qa_frame
        ),
        "retrieval": (
            "TF-IDF + latent semantic hybrid"
        ),
        "answering": (
            "deterministic evidence-grounded extraction"
        ),
        "seed": SEED,
        "offline_execution": True,
        "python": (
            platform.python_version()
        ),
    },
    name="Lesson 59 experiment",
)

# 50. Knowledge Check

1. What is extractive QA?
2. How does generative QA differ?
3. What is open-domain QA?
4. Why is retrieval important for factual QA?
5. What does evidence grounding mean?
6. Why attach citations?
7. What does Recall@k measure in QA retrieval?
8. What does MRR emphasize?
9. What is exact match?
10. What does token F1 measure?
11. Why should answer and retrieval errors be separated?
12. What is abstention?
13. Why is confidence calibration important?
14. Why must Arabic QA define a tashkeel policy?
15. What should a production QA system monitor?

# 51. Exercises

1. Add 20 more passages.
2. Expand the QA evaluation set.
3. Add a true BM25 retriever.
4. Add Reciprocal Rank Fusion.
5. Add a reranker.
6. Implement better answer-span extraction.
7. Add explicit unanswerable training examples.
8. Add multi-hop questions.
9. Add fully vocalized Arabic QA examples.
10. Build a final evaluation table with Recall@k, MRR, EM, F1, groundedness, citation validity, and latency.

## Challenge Exercises

1. Replace latent retrieval with SentenceTransformer embeddings.
2. Add a CrossEncoder reranker.
3. Use a pretrained extractive QA model.
4. Add a generative answer model with citations.
5. Compare closed-book, open-book, and RAG QA on the same evaluation set.

# 52. Summary and Next Lesson

In this lesson:

- extractive, generative, open-domain, and retrieval-augmented QA were distinguished;
- sparse and latent-semantic retrieval were combined;
- evidence recall was evaluated;
- answer-bearing sentences were selected;
- deterministic extractive answer generation was implemented;
- citations, confidence, and abstention were added;
- exact match, token F1, Recall@k, MRR, groundedness, and citation validity were computed;
- retrieval and answer failures were analyzed separately;
- multi-hop, long-context, multilingual, and Arabic QA were discussed;
- production latency, monitoring, safety, and calibration were connected to QA design.

## Next Lesson

**Lesson 60: Conversational AI and Dialogue Systems — Intent, State Tracking,
Response Generation, Memory, and Evaluation**

# References

- Rajpurkar, P. et al. work on SQuAD.
- Chen, D. et al. work on open-domain question answering.
- Karpukhin, V. et al. *Dense Passage Retrieval for Open-Domain Question Answering*.
- Lewis, P. et al. *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*.
- Research on calibration, abstention, and evidence-grounded QA.